In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from bs4 import BeautifulSoup
import pandas as pd
import time

In [8]:
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomaticControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0 Safari/537.36")

In [9]:
driver = webdriver.Chrome(options=options)
driver.get("https://food.ndtv.com/recipes/indian-recipes")

In [10]:
time.sleep(3)

In [11]:
while True:
    try:
        WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.src_pgn_lk a'))
        )
        print('Clicking "More Results"...')
        driver.execute_script("arguments[0].click();", more_button)
        time.sleep(3)
    except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
        print("No more 'More Results' button found.")
        break

No more 'More Results' button found.


In [12]:
html = driver.page_source
# print(html)

In [13]:
soup = BeautifulSoup(html, 'html.parser')
# print(soup.prettify)

In [14]:
recipes = soup.find_all('div', class_='SrcCrd-Rec')

In [15]:
all_data = []

for recipe in recipes:
    name_tag = recipe.find('span', class_='crd_lnk')
    link_tag = recipe.find('a', class_='crd_ttl', href=True)
    time_tag = recipe.find('div', class_='crd_ft_bt')

    if name_tag and link_tag:
        name = name_tag.text.strip()
        link = link_tag['href'].strip()
        time_taken = time_tag.text.strip() if time_tag else None
        
        all_data.append({'Dish Name': name, 
                         'Link': link,
                         'Time': time_taken
        })


# Save or display
df = pd.DataFrame(all_data)
print(df)

# Optionally save to CSV
df.to_csv("indian_dishes.csv", index=False)

                                    Dish Name  \
0   Rajasthani Lehsun Ka Halwa (Garlic Halwa)   
1                             Zero-Oil Ghugni   
2                        Malai Gulab Ki Kheer   
3                      Gucchi Pecan Nut Pulao   
4                           Tadka Pecan Raita   
..                                        ...   
67                               गाजर पायस्यम   
68                          इंस्टेंट सेट डोसा   
69                            मेथी पनीर पराठा   
70                          मैंगो कोकोनट चटनी   
71                        नारियल-अदरक की चटनी   

                                                 Link  Time  
0   https://food.ndtv.com/recipe-rajasthani-lehsun...  None  
1   https://food.ndtv.com/recipe-zero-oil-ghugni-9...  None  
2   https://food.ndtv.com/recipe-malai-gulab-ki-kh...  None  
3   https://food.ndtv.com/recipe-gucchi-pecan-nut-...  None  
4   https://food.ndtv.com/recipe-tadka-pecan-raita...  None  
..                                     

In [17]:
df

,Dish Name,Link,Time
0,Rajasthani Lehsun Ka Halwa (Garlic Halwa),https://food.ndtv.com/recipe-rajasthani-lehsun...,None
1,Zero-Oil Ghugni,https://food.ndtv.com/recipe-zero-oil-ghugni-9...,None
2,Malai Gulab Ki Kheer,https://food.ndtv.com/recipe-malai-gulab-ki-kh...,None
3,Gucchi Pecan Nut Pulao,https://food.ndtv.com/recipe-gucchi-pecan-nut-...,None
4,Tadka Pecan Raita,https://food.ndtv.com/recipe-tadka-pecan-raita...,None
...,...,...,...
67,गाजर पायस्यम,https://food.ndtv.com/recipe-carrot-payasam-hi...,None
68,इंस्टेंट सेट डोसा,https://food.ndtv.com/recipe-instant-set-dosa-...,None
69,मेथी पनीर पराठा,https://food.ndtv.com/recipe-methi-paneer-para...,None
70,मैंगो कोकोनट चटनी,https://food.ndtv.com/recipe-mango-coconut-chu...,None


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd

options = Options()
options.add_argument("--headless")  # optional
driver = webdriver.Chrome(options=options)

url = "https://food.ndtv.com/recipes"
driver.get(url)

wait = WebDriverWait(driver, 10)

# Keep clicking 'More Results' until it's gone
while True:
    try:
        more_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'More Results')]")))
        driver.execute_script("arguments[0].click();", more_button)
        time.sleep(3)  # wait for new results to load
    except:
        print("No more results button found.")
        break

# Now parse all visible results
soup = BeautifulSoup(driver.page_source, 'html.parser')
cards = soup.find_all("div", class_="src_rp_card")  # adjust if class differs

recipes = []
for card in cards:
    name = card.find("h2")
    link = card.find("a")
    time_tag = card.find("span", class_="cook-time")  # if available
    if name and link:
        recipes.append([name.text.strip(), link['href'], time_tag.text.strip() if time_tag else ''])

driver.quit()

df = pd.DataFrame(recipes, columns=["Dish Name", "Link", "Time"])
df.to_csv("ndtv_all_recipes.csv", index=False)
print(f"✅ Scraped {len(df)} recipes total!")


No more results button found.
✅ Scraped 0 recipes total!


In [16]:
driver.quit()